In [1]:
#Importar bibliotecas y cargar datos

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [2]:
data_0 = pd.read_csv('/datasets/geo_data_0.csv')
data_1 = pd.read_csv('/datasets/geo_data_1.csv')
data_2 = pd.read_csv('/datasets/geo_data_2.csv')

In [3]:
#Verificar datos y estrutura

for i, data in enumerate([data_0, data_1, data_2]):
    print(f"Región {i}")
    print(data.info())
    print(data.describe())
    print(data.isna().sum())
    print()

Región 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248      44.288691
min        -1.408605      -0.848218     -12.088328       0.000000
25%        -0.072580      -0.200881       0.287748      56.497507
50%         0.502360       0.250252       2.515969      91.849972
75%         1.073581       0.700646       4.715088     128.564089
max         2.362331    

Conclusión general sobre la estructura de los datos<br>
-Conjuntos completos: No hay valores ausentes (los datos están listos para modelado).<br>
-Misma estructura en las 3 regiones: Permite aplicar el mismo modelo en todos los conjuntos.<br>
-Diferencias importantes entre regiones:<br>
>Región 1: datos muy dispersos y menos confiables para predicción.<br>
Región 2: datos equilibrados y buena media de producción.<br>
Región 0: equilibrio aceptable y buenos resultados potenciales.


<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Muy bien hecha la revisión inicial de los datos para tu análisis, siempre es importante revisar la calidad de tus datos y realizar las transformaciones necesarias 
</div>

## 2-Entrena y prueba el modelo para cada región
Vamos a crear una función para repetir el mismo proceso en cada región.<br>
Función para:<br>
Separar los datos<br>
Entrenar regresión lineal<br>
Calcular previsión y error<br>
Retornar los valores importantes

In [4]:
def treinar_modelo(data, nome=''):
    print(f"1 Región: {nome}")
    
    # Separar features y target
    features = data.drop(['product', 'id'], axis=1)
    target = data['product']

    # Dividir datos (75% entrenamiento / 25% validación)
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=42
    )

    # Modelo
    modelo = LinearRegression()
    modelo.fit(X_train, y_train)

    # Predicciones
    previsao = modelo.predict(X_valid)

    # RMSE
    rmse = mean_squared_error(y_valid, previsao, squared=False)

    # promédio de las predicciones
    media_prevista = previsao.mean()
    media_real = y_valid.mean()

    print(f"2 RMSE: {rmse:.2f}")
    print(f"3 Promédio previsto: {media_prevista:.2f}")
    print(f"4 Promédio real: {media_real:.2f}")
    print("-" * 40)

    # Retorna para uso futuro
    return X_valid, y_valid, previsao

In [5]:
#Ejecutar para las 3 regiones
X0_valid, y0_valid, p0 = treinar_modelo(data_0, nome='Región 0')
X1_valid, y1_valid, p1 = treinar_modelo(data_1, nome='Región 1')
X2_valid, y2_valid, p2 = treinar_modelo(data_2, nome='Región 2')

1 Región: Región 0
2 RMSE: 37.76
3 Promédio previsto: 92.40
4 Promédio real: 92.33
----------------------------------------
1 Región: Región 1
2 RMSE: 0.89
3 Promédio previsto: 68.71
4 Promédio real: 68.73
----------------------------------------
1 Región: Región 2
2 RMSE: 40.15
3 Promédio previsto: 94.77
4 Promédio real: 95.15
----------------------------------------


<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Buen trabajo con los entrenamientos! Si bien los otros modelos pueden presentar volúmenes mucho mayores que las predicciones de la región 1 estas tienen un error de predicción más alto lo cual al momento de llevarlo a la vida real no resultará cómo en las predicciones, recuerda siempre tomar en cuenta la métrica de error de tu modelo para considerar cual es mejor y así ver en base a sus predicciones
</div>

## 3 – Preparar para cálculo de ganancias

Condiciones del problema:<br>
200 pozos desarrollados<br>
100 millones de dólares de inversión<br>
Cada unidad de producción genera US 4.500 dolares (porque cada unidad = 1000 barriles × $4.5/barril) <br>
Para no tener pérdidas: <br>

|Etapa |Cálculo          |Significado|
|---|---|---|
|1     ️|100.000.000 ÷ 200|Cada pozo cuesta US$ 500.000|
|2️     |500.000 ÷ 4.500  |Cada pozo debe produzir 111,1 unidades (mil barriles) para empatar la inversión|

In [6]:
# Comparar com promédios reales de cada región:

In [7]:

print("Comparando com producción mínima necesária (111.1 unidades):")
print(f"Región 0 – promédio real: {y0_valid.mean():.2f}")
print(f"Región 1 – promédio real: {y1_valid.mean():.2f}")
print(f"Región 2 – promédio real: {y2_valid.mean():.2f}")


Comparando com producción mínima necesária (111.1 unidades):
Región 0 – promédio real: 92.33
Región 1 – promédio real: 68.73
Región 2 – promédio real: 95.15


Conclusiones parciales<br>
Región 2 tiende a tener el promedio más alto de produção<br>
Región 1 tiene el más bajo RMSE<br>
Los datos están listos para simular el lucro y aplicar bootstrapping (parte 4)

## 4 – Cálculo directo de lucro con los 200 mejores pozos

In [8]:
#   Función para calcular lucro com los 200 mejores pozos

def calcular_lucro(y_valid, previsao, custo_total=100_000_000, preco_unitario=4500, poços=200):
    
    # Obtener índices de los 200 pozos com valor previsto más altos
    top_200_idx = previsao.argsort()[-poços:]
    
    # Obtener volume real de eses pozos (no la previsão)
    volume_real = y_valid.iloc[top_200_idx]
    
    # Calcular la ganancia total
    lucro_total = volume_real.sum() * preco_unitario - custo_total
    
    # Promédio de produção de los 200 mejores
    media_volume = volume_real.mean()
    
    return lucro_total, media_volume, volume_real

In [9]:
lucro_0, media_0, top200_0 = calcular_lucro(y0_valid, p0)
lucro_1, media_1, top200_1 = calcular_lucro(y1_valid, p1)
lucro_2, media_2, top200_2 = calcular_lucro(y2_valid, p2)

print(f"Región 0 - Lucro: ${lucro_0:,.2f} | Promédio: {media_0:.2f}")
print(f"Región 1 - Lucro: ${lucro_1:,.2f} | Promédio: {media_1:.2f}")
print(f"Región 2 - Lucro: ${lucro_2:,.2f} | Promédio: {media_2:.2f}")

Región 0 - Lucro: $33,591,411.14 | Promédio: 148.43
Región 1 - Lucro: $24,150,866.97 | Promédio: 137.95
Región 2 - Lucro: $25,985,717.59 | Promédio: 139.98


#Com base en los 200 mejores pozos y no en el lucro estimado directo, escogemos la región con ganancias líquidas más grandes.
Región recomendada : Región 0 (más grande lucro entre todas).

<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Muy bien realizados los cálculos para determinar las ganancias próximas de cada región en base a los modelos entrenados, consideraste muy bien la métrica de error y los cálculos para determinar la región que sería más adecuada para la inversión
</div>

## 5 – Cálculo de  riesgos y ganancias para cada región (Bootstrapping: riesgo y intervalo de confianza)

#Vamos simular 1.000 muestras aleatórias con reposición de 200 pozos de cada región y calcular lucro en cada simulación para construir:<br>
-Distribuición de lucros<br>
-Intervalo de confianza 95%<br>
-Probabilidad de pérdida (lucro < 0)

In [10]:
#Función de Bootstrapping

def simular_bootstrap(amostras, preco_unitario=4500, custo_total=100_000_000, n_sim=1000, poços=200):
    lucros = []
    
    for _ in range(n_sim):
        sample = amostras.sample(n=poços, replace=True)
        lucro = sample.sum() * preco_unitario - custo_total
        lucros.append(lucro)
    
    lucros = pd.Series(lucros)
    
    # Cálculos estadísticos
    media = lucros.mean()
    intervalo = lucros.quantile([0.025, 0.975])
    risco_perda = (lucros < 0).mean() * 100  # em %

    return media, intervalo, risco_perda, lucros

In [11]:

#Aplicar para cada región
media_b0, ic_b0, risco_b0, dist_b0 = simular_bootstrap(top200_0)
media_b1, ic_b1, risco_b1, dist_b1 = simular_bootstrap(top200_1)
media_b2, ic_b2, risco_b2, dist_b2 = simular_bootstrap(top200_2)

print(f"Región 0 - Promédio: ${media_b0:,.2f} | IC95%: {ic_b0.values} | Riesgo: {risco_b0:.2f}%")
print(f"Región 1 - Promédio: ${media_b1:,.2f} | IC95%: {ic_b1.values} | Riesgo: {risco_b1:.2f}%")
print(f"Región 2 - Promédio: ${media_b2:,.2f} | IC95%: {ic_b2.values} | Riesgo: {risco_b2:.2f}%")


Región 0 - Promédio: $33,680,799.01 | IC95%: [30632874.10377941 36572355.20453463] | Riesgo: 0.00%
Región 1 - Promédio: $24,150,866.97 | IC95%: [24150866.96681511 24150866.96681511] | Riesgo: 0.00%
Región 2 - Promédio: $25,925,518.94 | IC95%: [22368748.72104415 29606436.88181241] | Riesgo: 0.00%


## Conclusión Final
Critérios para escoger la mejor región:

1. Riesgo < 2.5%
2. Promédio más alto de ganancia
3. Intervalo de confianza más estable (como critério secundário)
   
✔️ Región recomendada final: Región 0, devido a tener el más alto promédio de ganancia ($33,597,549.49) y con riesgo inferior a 2.5%.

<div class="alert alert-block alert-success">
<b>Comentario del revisor (1ra Iteracion)</b> <a class=“tocSkip”></a>

Buen trabajo Miguel! Analizaste correctamente los resultados, cuando se hace este tipo de aplicaciones con modelos predictivos siempre hay que tener en cuenta la métrica de error ya que si tenemos un modelo que puede mostrar más beneficio pero su métrica de error es alta quiere decir que cuando sea aplicado puede que no se acerque a ese beneficio.
    
Saludos!
</div>